# Efficient and Reproducible Biomedical Question Answering using RAG
## Reproduction of IEEE SDS 2025 Paper

---

# 1. Install Dependencies

---

# 2. Import Libraries

---

# 3. Load BioASQ Dataset

---

# 4. Explore the Dataset

In [1]:
!pip install datasets
!pip install pandas
!pip install numpy
!pip install matplotlib

  Using cached huggingface_hub-1.26.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.26.0-py3-none-any.whl (780 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.22.2
    Uninstalling huggingface-hub-0.22.2:
      Successfully uninstalled huggingface-hub-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.26.0 which is incompatible.
transformers 4.40.0 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.26.0 which is incompatible.


In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
!git clone https://github.com/slinusc/medical_RAG_system.git

fatal: destination path 'medical_RAG_system' already exists and is not an empty directory.


In [4]:
%cd medical_RAG_system

/content/medical_RAG_system


In [5]:
!pwd

/content/medical_RAG_system


In [6]:
!ls

biobert_embeddings_10k.npy  LICENSE		       requirements.txt
evaluation		    medcpt_embeddings_10k.npy  sys_requirements.txt
filtered_bioasq.json	    rag_system
information_retrieval	    README.md


In [7]:
!head -100 README.md


# Medical RAG System

This repository contains a comprehensive implementation of a Medical Retrieval-Augmented Generation (RAG) system. The system integrates multiple components for document retrieval, question answering, and evaluation, tailored specifically for the medical domain.

## Table of Contents
- [Overview](#overview)
- [File Structure](#file-structure)
- [Installation](#installation)
- [Usage](#usage)
- [Components](#components)
  - [Retrieval System](#retrieval-system)
  - [Question Answering System](#question-answering-system)
  - [Evaluation](#evaluation)
  - [Data Storage](#data-storage)
- [Contributing](#contributing)
- [License](#license)

## Overview

The Medical RAG System is designed to enhance medical information retrieval and provide accurate answers to medical queries. It combines various retrieval methods, including BM25, bioBERT, and hybrid models, with advanced question-answering techniques to ensure precise and relevant results.


## File structure

```plain

In [8]:
!cat requirements.txt

anaconda==0.0.1.1  # Anaconda package
annotated-types==0.6.0  # Support for typing-annotations
anyio==4.3.0  # Async network and file operations
argon2-cffi==23.1.0  # The secure Argon2 password hashing algorithm
attrs==23.2.0  # Attributes without boilerplate
Babel==2.14.0  # Internationalization utilities
beautifulsoup4==4.12.3  # Screen-scraping library
bleach==6.1.0  # Sanitize your inputs
click==8.1.7  # Command Line Interface Creation Kit
decorator==5.1.1  # Simplifies the usage of decorators
elastic-transport==8.13.0  # Transport layer for Elasticsearch
elasticsearch==8.13.0  # Official Elasticsearch client
faiss-cpu==1.8.0  # A library for efficient similarity search and clustering
Flask==3.0.3  # Micro web framework
fsspec==2024.3.1  # File system specification
huggingface-hub==0.22.2  # Client library for Huggingface hub
idna==3.3  # Internationalized Domain Names in Applications (IDNA)
importlib-metadata==4.6.4  # Library to access the metadata for a Python package
joblib==1

In [9]:
import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("No GPU Found!")

PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4


In [10]:
!pip install -q \
sentence-transformers==2.7.0 \
transformers==4.40.0 \
faiss-cpu==1.8.0 \
rank_bm25 \
datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [11]:
!pip install -q datasets

In [12]:
from datasets import load_dataset

In [13]:
pubmed = load_dataset(
    "slinusc/PubMedAbstractsSubset",
    split="train",
    streaming=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

In [14]:
first_doc = next(iter(pubmed))

In [15]:
print(first_doc.keys())

dict_keys(['title', 'abstract', 'PMID'])


In [16]:
print("PMID:")
print(first_doc["PMID"])

print("\nTitle:")
print(first_doc["title"])

print("\nAbstract:")
print(first_doc["abstract"])

PMID:
22

Title:
[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].

Abstract:
A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without hyperglycemia in its most efficient dosage of 47 percent LD50 (150

In [17]:
!find . -type f | grep -i "bioasq"

./filtered_bioasq.json


In [18]:
!find . -type f | grep -i "json"

./filtered_bioasq.json
./information_retrieval/elastic_container/errors.jsonl


In [19]:
!grep -Rin "BioASQ" .

./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:185:    "rag_type.analyze_performance(\"/home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min2/result_ragver_4.json\")"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:204:      "Directory 'experiment_bioASQ_min1_bioBERT' already exists at /home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min1_bioBERT\n"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:209:    "experiment_name = \"experiment_bioASQ_min1_bioBERT\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:213:    "question_input_factoid = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/factoid_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:214:    "question_input_summary = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/summary_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:215:    "question_input_list = \

In [20]:
!grep -Rin "bioasq" .

./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:185:    "rag_type.analyze_performance(\"/home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min2/result_ragver_4.json\")"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:204:      "Directory 'experiment_bioASQ_min1_bioBERT' already exists at /home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min1_bioBERT\n"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:209:    "experiment_name = \"experiment_bioASQ_min1_bioBERT\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:213:    "question_input_factoid = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/factoid_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:214:    "question_input_summary = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/summary_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:215:    "question_input_list = \

In [21]:
!grep -Rin "questions" .

./filtered_bioasq.json:2:  "questions": [
./rag_system/openAI_chat.py:17:            "documents to answer questions. The first documents should be the most relevant."
./rag_system/openAI_chat.py:19:            "When answering questions, always format your response "
./rag_system/openAI_chat.py:22:            "Please think step-by-step before answering questions and provide the most accurate response possible."
./rag_system/pipeline.ipynb:497:      "        \"content\": \"Attention Deficit Hyperactivity Disorder (ADHD) is a common neurobehavioral problem in children that the medical practitioner is frequently asked to diagnose and treat. Equally as important as accurate diagnosis and treatment, however, is the ability to provide family members with clear and concise information that leads to an understanding of the disorder. This article presents a framework for answering family members' specific questions about ADHD and recommendations for ways to effectively share information with fam

In [22]:
!jupyter nbconvert \
--to script \
evaluation/evaluation_QA_system/dataset_filter/filter_data.ipynb

[NbConvertApp] Converting notebook evaluation/evaluation_QA_system/dataset_filter/filter_data.ipynb to script
[NbConvertApp] Writing 20655 bytes to evaluation/evaluation_QA_system/dataset_filter/filter_data.py


In [23]:
!head -250 evaluation/evaluation_QA_system/dataset_filter/filter_data.py

#!/usr/bin/env python
# coding: utf-8

# # Filter dataset

# first we loop trough each training set for example BioASQ-trainingDataset2b.json and extract the pubmed IDS used to answers questions 

# In[1]:


import os
import json
import pandas as pd
from tqdm import tqdm

# Define the directories
json_dir = '~/Questions_answers_data/DATEN_RAG_PM4/trainings_sets'
csv_dir = os.path.expanduser(json_dir + '/csv')  # Ensure the path is expanded to the user's home directory

# Create the CSV directory if it doesn't exist
os.makedirs(csv_dir, exist_ok=True)

# Initialize a set to hold all unique PubMed IDs across files
all_pubmed_ids = set()

# List all JSON files in the directory
json_files = [f for f in os.listdir(os.path.expanduser(json_dir)) if f.endswith('.json')]  # Ensure the path is expanded

# Loop through files with a tqdm progress bar
for json_file in tqdm(json_files, desc="Processing JSON Files"):
    json_path = os.path.join(os.path.expanduser(json_dir), json_file)

    # Load JS

# BM25 Retrieval

In [24]:
!pip install -q rank_bm25

In [25]:
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from tqdm import tqdm
import re

In [26]:
pubmed = load_dataset(
    "slinusc/PubMedAbstractsSubset",
    split="train",
    streaming=True
)

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

In [27]:
documents = []

for i, doc in enumerate(pubmed):

    documents.append(doc)

    if i == 9999:
        break

print("Number of documents:", len(documents))

Number of documents: 10000


In [28]:
corpus = []

for doc in documents:

    text = doc["title"] + " " + doc["abstract"]

    corpus.append(text)

In [29]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

In [30]:
tokenized_corpus = [
    preprocess(doc)
    for doc in corpus
]

In [31]:
tokenized_corpus[0]

['demonstration',
 'of',
 'tumor',
 'inhibiting',
 'properties',
 'of',
 'a',
 'strongly',
 'immunostimulating',
 'lowmolecular',
 'weight',
 'substance',
 'comparative',
 'studies',
 'with',
 'ifosfamide',
 'on',
 'the',
 'immunolabile',
 'ds',
 'carcinosarcoma',
 'stimulation',
 'of',
 'the',
 'autoimmune',
 'activity',
 'for',
 'approx',
 '20',
 'days',
 'by',
 'ba',
 '1',
 'a',
 'n2cyanoethyleneurea',
 'novel',
 'prophylactic',
 'possibilities',
 'a',
 'report',
 'is',
 'given',
 'on',
 'the',
 'recent',
 'discovery',
 'of',
 'outstanding',
 'immunological',
 'properties',
 'in',
 'ba',
 '1',
 'n2cyanoethyleneurea',
 'having',
 'a',
 'low',
 'molecular',
 'mass',
 'm',
 '111104',
 'experiments',
 'in',
 '214',
 'ds',
 'carcinosarcoma',
 'bearing',
 'wistar',
 'rats',
 'have',
 'shown',
 'that',
 'ba',
 '1',
 'at',
 'a',
 'dosage',
 'of',
 'only',
 'about',
 '12',
 'percent',
 'ld50',
 '150',
 'mg',
 'kg',
 'and',
 'negligible',
 'lethality',
 '17',
 'percent',
 'results',
 'in',
 '

In [32]:
bm25 = BM25Okapi(tokenized_corpus)

In [33]:
documents[0]

{'title': '[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].',
 'abstract': 'A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without hyperglycemia in its most efficient dosage of 47 percent LD50 (150 

In [34]:
tokenized_corpus[0][:30]

['demonstration',
 'of',
 'tumor',
 'inhibiting',
 'properties',
 'of',
 'a',
 'strongly',
 'immunostimulating',
 'lowmolecular',
 'weight',
 'substance',
 'comparative',
 'studies',
 'with',
 'ifosfamide',
 'on',
 'the',
 'immunolabile',
 'ds',
 'carcinosarcoma',
 'stimulation',
 'of',
 'the',
 'autoimmune',
 'activity',
 'for',
 'approx',
 '20',
 'days']

In [35]:
query = "What is the treatment for diabetes?"

In [36]:
tokenized_query = preprocess(query)

print(tokenized_query)

['what', 'is', 'the', 'treatment', 'for', 'diabetes']


In [37]:
scores = bm25.get_scores(tokenized_query)

In [38]:
import numpy as np

top_n = 50

top_indices = np.argsort(scores)[::-1][:top_n]

In [39]:
np.argsort(scores)

array([9764, 3069, 7910, ..., 5207, 6764, 6469])

In [40]:
for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)
    print(f"Rank {rank}")
    print(f"Score : {scores[idx]:.3f}")
    print(f"PMID  : {documents[idx]['PMID']}")
    print(f"Title : {documents[idx]['title']}")

Rank 1
Score : 20.537
PMID  : 137164
Title : [Incidence and manifestations of the heterozygosity of the gene causing diabetes].
Rank 2
Score : 19.322
PMID  : 143916
Title : Islet transplantation in genetically determined diabetes.
Rank 3
Score : 18.598
PMID  : 111165
Title : [Current status in transplantation of the endocrine pancreas in the treatment of diabetes mellitus 2. Transplantation of fetal pancreas].
Rank 4
Score : 18.264
PMID  : 83349
Title : The tumor-producing effect of automobile exhaust condensate and fractions thereof. Part III: mathematical-statistical evaluation of the test results.
Rank 5
Score : 18.114
PMID  : 99735
Title : [Enlarge indications for controlled respiration (author's transl)].
Rank 6
Score : 17.775
PMID  : 47532
Title : Association of foot lesions with retinopathy in patients with newly diagnosed diabetes.
Rank 7
Score : 17.722
PMID  : 69935
Title : Familial studies of type-I and type-II idiopathic diabetes mellitus.
Rank 8
Score : 17.435
PMID  : 41691

In [41]:
import re

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were",
    "of", "to", "for", "in", "on", "at", "with",
    "what", "which", "who", "when", "where", "why",
    "how", "and", "or"
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)

    tokens = text.split()

    tokens = [
        word
        for word in tokens
        if word not in STOPWORDS
    ]

    return tokens

In [42]:
preprocess("What is the treatment for diabetes?")

['treatment', 'diabetes']

In [43]:
tokenized_corpus = [
    preprocess(doc)
    for doc in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

In [65]:
question = bioasq["questions"][0]
query = question["body"]
tokenized_query = preprocess(query)

scores = bm25.get_scores(tokenized_query)

top_indices = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top_indices, start=1):

    print("="*70)
    print(f"Rank {rank}")
    print(f"Score : {scores[idx]:.2f}")
    print(f"PMID  : {documents[idx]['PMID']}")
    print(f"Title : {documents[idx]['title']}")

Rank 1
Score : 12.87
PMID  : 202219
Title : Dietary and serum lipids in the multifactorial etiology of atherosclerosis.
Rank 2
Score : 11.71
PMID  : 112034
Title : Another family with purine neucleoside phosphorylase deficiency.
Rank 3
Score : 11.01
PMID  : 103250
Title : The Montgomery lecture, 1977. Curious colobomata.
Rank 4
Score : 10.33
PMID  : 46695
Title : Limitations of the usefulness of the d-xylose absorption test.
Rank 5
Score : 9.96
PMID  : 131223
Title : A multifactorial system controlling myeloid cell differentiation and division.


BIOBERT RETRIVAL

Question -> BioBERT Encoder -> Question Embedding -> Compare with Document Embeddings -> Top-k Documents

In [66]:
!pip install -q sentence-transformers

In [67]:
from sentence_transformers import SentenceTransformer
import numpy as np

# BioBERT Semantic Retrieval

In [68]:
from sentence_transformers import SentenceTransformer, models
import torch

In [69]:
device = "cuda" if torch.cuda.is_available() else "cpu"

word_embedding_model = models.Transformer(
    "dmis-lab/biobert-v1.1",
    max_seq_length=512
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

biobert = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [70]:
sentence = "Metformin is used to treat diabetes."

embedding = biobert.encode(sentence)

print(embedding.shape)

(768,)


In [71]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [72]:
corpus = []

for doc in documents:
    text = doc["title"] + " " + doc["abstract"]
    corpus.append(text)

print(corpus[0][:200])

[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of


In [73]:
from sentence_transformers import SentenceTransformer, models
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

word_embedding_model = models.Transformer(
    "dmis-lab/biobert-v1.1",
    max_seq_length=512
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)

biobert = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [74]:
sentence = "Metformin is used to treat diabetes."

embedding = biobert.encode(sentence)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(768,)


In [75]:
test_embeddings = biobert.encode(
    corpus[:10],
    show_progress_bar=True,
    convert_to_numpy=True
)

print(test_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(10, 768)


In [76]:
print(embedding.shape)

(768,)


In [77]:
print(test_embeddings.shape)

(10, 768)


In [78]:
import numpy as np

document_embeddings = biobert.encode(
    corpus,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding Shape:", document_embeddings.shape)

# Save embeddings
np.save("biobert_embeddings_10k.npy", document_embeddings)

print("Embeddings saved successfully!")

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embedding Shape: (10000, 768)
Embeddings saved successfully!


In [79]:
print(document_embeddings.shape)

(10000, 768)


In [80]:
document_embeddings[:2]

array([[ 0.02567425, -0.19570793, -0.15966554, ...,  0.25370413,
        -0.09315991,  0.00287652],
       [ 0.07904429,  0.00125223, -0.1261476 , ...,  0.29271623,
         0.00890275,  0.03762019]], dtype=float32)

In [81]:
question = bioasq["questions"][0]
query = question["body"]

query_embedding = biobert.encode(
    query,
    convert_to_numpy=True
)

print(query_embedding.shape)

(768,)


In [82]:
from sklearn.metrics.pairwise import cosine_similarity

In [83]:
similarities = cosine_similarity(
    [query_embedding],
    document_embeddings
)[0]

In [84]:
import numpy as np

top_k = 5

top_indices = np.argsort(similarities)[::-1][:top_k]

In [85]:
for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Similarity: {similarities[idx]:.4f}")
    print(f"PMID: {documents[idx]['PMID']}")
    print(f"Title: {documents[idx]['title']}")

Rank: 1
Similarity: 0.9053
PMID: 119701
Title: Hurler-Scheie phenotype: a report of two pairs of inbred sibs.
Rank: 2
Similarity: 0.9045
PMID: 148839
Title: Hereditary atrial septal defect. Update of a large kindred.
Rank: 3
Similarity: 0.9035
PMID: 112034
Title: Another family with purine neucleoside phosphorylase deficiency.
Rank: 4
Similarity: 0.9017
PMID: 117710
Title: Dyggve-Melchior-Clausen syndrome: genetic studies and report of affected sibs.
Rank: 5
Similarity: 0.8969
PMID: 181111
Title: Van Buchem's disease (hyperostosis corticalis generalisata)


In [86]:

count = 0

for doc in documents:
    text = (doc["title"] + " " + doc["abstract"]).lower()

    if "diabetes" in text:
        count += 1

print("Documents mentioning diabetes:", count)

Documents mentioning diabetes: 89


In [48]:
!pip install -q datasets

In [49]:
from datasets import load_dataset

bioasq = load_dataset("jmhb/BioASQ")

In [50]:
print(bioasq["factoid"][0])

{'type': 'factoid', 'question': 'Which thyroid hormone transporter is implicated in thyroid hormone resistance syndrome?', 'answer': "['TH monocarboxylate transporter 8 (MCT8) mutation is implicated in the TH resistance syndrome']", 'ideal_answer': 'Hemizygous MCT8 mutations cuases TH resistance syndrome in males characterized by severe psychomotor retardation, known as the Allan-Herndon-Dudley syndrome (AHDS).', 'documents': ['http://www.ncbi.nlm.nih.gov/pubmed/23392090', 'http://www.ncbi.nlm.nih.gov/pubmed/22986150', 'http://www.ncbi.nlm.nih.gov/pubmed/21459689', 'http://www.ncbi.nlm.nih.gov/pubmed/18940949', 'http://www.ncbi.nlm.nih.gov/pubmed/17574009', 'http://www.ncbi.nlm.nih.gov/pubmed/17161330', 'http://www.ncbi.nlm.nih.gov/pubmed/1422238', 'http://www.ncbi.nlm.nih.gov/pubmed/8475937', 'http://www.ncbi.nlm.nih.gov/pubmed/9092799', 'http://www.ncbi.nlm.nih.gov/pubmed/21874823', 'http://www.ncbi.nlm.nih.gov/pubmed/19541799', 'http://www.ncbi.nlm.nih.gov/pubmed/12946875', 'http://

In [51]:
!unzip BioASQ-training13b.zip

unzip:  cannot find or open BioASQ-training13b.zip, BioASQ-training13b.zip.zip or BioASQ-training13b.zip.ZIP.


In [52]:
!ls


biobert_embeddings_10k.npy  LICENSE		       requirements.txt
evaluation		    medcpt_embeddings_10k.npy  sys_requirements.txt
filtered_bioasq.json	    rag_system
information_retrieval	    README.md


In [53]:
!find . -name "*.zip"

In [54]:
!find /content -name "BioASQ-training13b.zip"

/content/BioASQ-training13b.zip


In [55]:
!unzip /content/BioASQ-training13b.zip -d /content/BioASQ

Archive:  /content/BioASQ-training13b.zip
replace /content/BioASQ/BioASQ-training13b/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/BioASQ/BioASQ-training13b/README  
replace /content/BioASQ/BioASQ-training13b/training13b.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/BioASQ/BioASQ-training13b/training13b.json  


In [56]:
!find /content/BioASQ -type f

/content/BioASQ/BioASQ-training13b/training13b.json
/content/BioASQ/BioASQ-training13b/README


In [57]:
import json

bioasq_path = "/content/BioASQ/BioASQ-training13b/training13b.json"

with open(bioasq_path, "r") as f:
    bioasq = json.load(f)

print("Number of questions:", len(bioasq["questions"]))

Number of questions: 5389


In [58]:
question = bioasq["questions"][0]

print(question.keys())

dict_keys(['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets'])


In [59]:
print("="*80)

print("Question:")
print(question["body"])

print("\nType:")
print(question["type"])

print("\nGround Truth Documents:")
print(question["documents"][:5])

print("\nIdeal Answer:")
print(question["ideal_answer"])

Question:
Is Hirschsprung disease a mendelian or a multifactorial disorder?

Type:
summary

Ground Truth Documents:
['http://www.ncbi.nlm.nih.gov/pubmed/15858239', 'http://www.ncbi.nlm.nih.gov/pubmed/20598273', 'http://www.ncbi.nlm.nih.gov/pubmed/6650562', 'http://www.ncbi.nlm.nih.gov/pubmed/12239580', 'http://www.ncbi.nlm.nih.gov/pubmed/21995290']

Ideal Answer:
["Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."]


In [60]:
ground_truth_pmids = [
    int(url.split("/")[-1])
    for url in question["documents"]
]

print(ground_truth_pmids)

[15858239, 20598273, 6650562, 12239580, 21995290, 23001136, 15617541, 8896569, 15829955]


In [61]:
query = question["body"]

In [62]:
retrieved_pmids = []

for idx in top_indices:
    retrieved_pmids.append(documents[idx]["PMID"])

print(retrieved_pmids)

[137164, 143916, 111165, 83349, 99735, 47532, 69935, 41691, 96000, 74564, 157515, 85645, 50109, 17585, 118410, 73362, 5754, 46337, 98676, 131001, 106020, 141784, 37977, 18032, 82784, 55014, 113974, 23608, 172171, 126887, 193114, 162266, 46338, 81522, 141929, 180186, 111140, 83675, 149490, 196988, 103606, 93031, 88441, 115340, 51140, 61982, 37498, 43436, 59936, 154487]


In [63]:
retrieved_pmids = []

for idx in top_indices:
    retrieved_pmids.append(documents[idx]["PMID"])

correct = set(ground_truth_pmids)

retrieved = set(retrieved_pmids)

hits = correct.intersection(retrieved)

print("Correct PMIDs:", len(correct))
print("Retrieved Correct:", len(hits))
print("Hits:", hits)

Correct PMIDs: 9
Retrieved Correct: 0
Hits: set()


In [64]:
dataset_pmids = set(doc["PMID"] for doc in documents)

available_pmids = []

for pmid in ground_truth_pmids:
    if pmid in dataset_pmids:
        available_pmids.append(pmid)

print("Ground truth PMIDs:", len(ground_truth_pmids))
print("Available in our corpus:", len(available_pmids))
print("PMIDs found:", available_pmids)

Ground truth PMIDs: 9
Available in our corpus: 0
PMIDs found: []


### MedCPT Reranking of BM25 Results

In [87]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"

query_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
).to(device)

query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)

# Encode the current BioASQ query using MedCPT Query Encoder
inputs = query_tokenizer(
    query,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    medcpt_query_embedding = query_model(**inputs).last_hidden_state[:, 0, :]

# Get the MedCPT embeddings for the documents retrieved by BM25
bm25_retrieved_embeddings = document_embeddings[top_indices]

# Calculate cosine similarities between the MedCPT query embedding and BM25 retrieved document embeddings
reranked_similarities = cosine_similarity(
    medcpt_query_embedding.cpu().numpy(),
    bm25_retrieved_embeddings
)[0]

# Get indices of the top documents after reranking
reranked_top_indices_in_bm25_results = reranked_similarities.argsort()[::-1]

print("MedCPT Reranked Top 5 from BM25's Top 50:")
for rank, rerank_idx in enumerate(reranked_top_indices_in_bm25_results[:5], start=1):
    original_doc_idx = top_indices[rerank_idx] # Get the original index in the full corpus

    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Reranked Similarity: {reranked_similarities[rerank_idx]:.4f}")
    print(f"PMID: {documents[original_doc_idx]['PMID']}")
    print(f"Title: {documents[original_doc_idx]['title']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MedCPT Reranked Top 5 from BM25's Top 50:
Rank: 1
Reranked Similarity: -0.0263
PMID: 148839
Title: Hereditary atrial septal defect. Update of a large kindred.
Rank: 2
Reranked Similarity: -0.0307
PMID: 117710
Title: Dyggve-Melchior-Clausen syndrome: genetic studies and report of affected sibs.
Rank: 3
Reranked Similarity: -0.0328
PMID: 119701
Title: Hurler-Scheie phenotype: a report of two pairs of inbred sibs.
Rank: 4
Reranked Similarity: -0.0349
PMID: 112034
Title: Another family with purine neucleoside phosphorylase deficiency.
Rank: 5
Reranked Similarity: -0.0435
PMID: 181111
Title: Van Buchem's disease (hyperostosis corticalis generalisata)


In [88]:
# Evaluate reranked results
retrieved_pmids_reranked = []

for rerank_idx in reranked_top_indices_in_bm25_results:
    original_doc_idx = top_indices[rerank_idx]
    retrieved_pmids_reranked.append(documents[original_doc_idx]["PMID"])

correct = set(ground_truth_pmids)
retrieved = set(retrieved_pmids_reranked)

hits = correct.intersection(retrieved)

print("BM25 + MedCPT Reranked Correct PMIDs:", len(correct))
print("BM25 + MedCPT Reranked Retrieved Correct:", len(hits))
print("Hits:", hits)

BM25 + MedCPT Reranked Correct PMIDs: 9
BM25 + MedCPT Reranked Retrieved Correct: 0
Hits: set()


# MedCPT Retrival

1. Install packages
2. Load MedCPT model
3. Encode 10k papers
4. Save embeddings
5. Encode query
6. Retrieve with FAISS
7. Compare with BioBERT

In [89]:
!pip uninstall -y numpy faiss-cpu
!pip install -q numpy==1.26.4 faiss-cpu==1.8.0
!pip install -q sentence-transformers transformers

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: faiss-cpu 1.8.0
Uninstalling faiss-cpu-1.8.0:
  Successfully uninstalled faiss-cpu-1.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompat

In [90]:
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer

In [91]:
model = SentenceTransformer(
    "ncbi/MedCPT-Article-Encoder"
)

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded!


In [92]:
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Article-Encoder"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Article-Encoder"
)

print(device)

cuda


In [93]:
sample = documents[0]["title"] + " " + documents[0]["abstract"]

inputs = tokenizer(
    sample,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

embedding = outputs.last_hidden_state[:,0,:]

print(embedding.shape)

torch.Size([1, 768])


In [94]:
corpus = [
    doc["title"] + " " + doc["abstract"]
    for doc in documents
]

print("Number of documents:", len(corpus))
print("\nFirst document:\n")
print(corpus[0][:300])

Number of documents: 10000

First document:

[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic


In [95]:
from tqdm import tqdm
import numpy as np

batch_size = 32

document_embeddings = []

model.eval()

for i in tqdm(range(0, len(corpus), batch_size)):

    batch = corpus[i:i+batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state[:, 0, :]

    document_embeddings.append(
        embeddings.cpu().numpy()
    )

document_embeddings = np.vstack(document_embeddings)

print(document_embeddings.shape)

100%|██████████| 313/313 [04:46<00:00,  1.09it/s]

(10000, 768)


In [96]:
np.save(
    "medcpt_embeddings_10k.npy",
    document_embeddings
)

print("Saved successfully!")

Saved successfully!


In [97]:
from transformers import AutoTokenizer, AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

query_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
).to(device)

query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)

print("Query encoder loaded!")

Query encoder loaded!


In [98]:
question = "What gene causes Alzheimer's disease?"

inputs = query_tokenizer(
    question,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    outputs = query_model(**inputs)

query_embedding = outputs.last_hidden_state[:,0,:]

print(query_embedding.shape)

torch.Size([1, 768])


In [99]:
from sklearn.metrics.pairwise import cosine_similarity

In [100]:
similarities = cosine_similarity(
    query_embedding.cpu().numpy(),
    document_embeddings
)

print(similarities.shape)

(1, 10000)


In [101]:
top_k = 5

top_indices = similarities[0].argsort()[-top_k:][::-1]

print(top_indices)

[4009 5074 5830 8515 5934]


In [102]:
for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)

    print(f"Rank      : {rank}")
    print(f"Similarity: {similarities[0][idx]:.4f}")
    print(f"PMID      : {documents[idx]['PMID']}")
    print(f"Title     : {documents[idx]['title']}")

Rank      : 1
Similarity: 0.5814
PMID      : 87027
Title     : HLA haplotype associations with disease.
Rank      : 2
Similarity: 0.5788
PMID      : 108483
Title     : Neurofibrillary pathology: current status and research perspectives.
Rank      : 3
Similarity: 0.5713
PMID      : 123591
Title     : A family with apparently sex-linked optic atrophy.
Rank      : 4
Similarity: 0.5681
PMID      : 178032
Title     : Genetic determination of aggressive behavior and brain cyclic AMP.
Rank      : 5
Similarity: 0.5646
PMID      : 126052
Title     : Alzheimer degeneration in Down syndrome. Electrophysiologic alterations and histopathologic findings.


# HYBRID APPROACH FOR RETRIVAL

01 Load Data

02 BM25 Retriever

03 MedCPT Reranker

04 Hybrid Retrieval

05 Evaluation

06 Visualization

In [103]:
question = bioasq["questions"][0]

query = question["body"]

ground_truth_pmids = [
    int(url.split("/")[-1])
    for url in question["documents"]
]

print(query)
print("\nGround Truth PMIDs:")
print(ground_truth_pmids)

Is Hirschsprung disease a mendelian or a multifactorial disorder?

Ground Truth PMIDs:
[15858239, 20598273, 6650562, 12239580, 21995290, 23001136, 15617541, 8896569, 15829955]


In [104]:
# BM25 Retrieval (Top-50)

top_k = 50

# Tokenize the BioASQ question
tokenized_query = preprocess(query)

# Get BM25 scores for all documents
scores = bm25.get_scores(tokenized_query)

# Sort scores in descending order
top_indices = scores.argsort()[-top_k:][::-1]

print(f"Retrieved {len(top_indices)} documents.")

Retrieved 50 documents.


In [105]:
for rank, idx in enumerate(top_indices[:10], start=1):

    print("="*80)
    print(f"Rank  : {rank}")
    print(f"Score : {scores[idx]:.4f}")
    print(f"PMID  : {documents[idx]['PMID']}")
    print(f"Title : {documents[idx]['title']}")

Rank  : 1
Score : 12.8710
PMID  : 202219
Title : Dietary and serum lipids in the multifactorial etiology of atherosclerosis.
Rank  : 2
Score : 11.7115
PMID  : 112034
Title : Another family with purine neucleoside phosphorylase deficiency.
Rank  : 3
Score : 11.0091
PMID  : 103250
Title : The Montgomery lecture, 1977. Curious colobomata.
Rank  : 4
Score : 10.3321
PMID  : 46695
Title : Limitations of the usefulness of the d-xylose absorption test.
Rank  : 5
Score : 9.9569
PMID  : 131223
Title : A multifactorial system controlling myeloid cell differentiation and division.
Rank  : 6
Score : 9.8535
PMID  : 114742
Title : [Pseudodiverticulosis of the esophagus (author's transl)].
Rank  : 7
Score : 9.7790
PMID  : 192732
Title : Nuclear mutations affecting mitochondrial structure and function in Chlamydomonas.
Rank  : 8
Score : 9.4161
PMID  : 91836
Title : Reduction in sudden deaths by a multifactorial intervention programme after acute myocardial infarction.
Rank  : 9
Score : 9.3392
PMID  : 1

In [106]:
print(type(top_indices))
print(len(top_indices))

<class 'numpy.ndarray'>
50


In [107]:
bm25_docs = [documents[idx] for idx in top_indices]
bm25_embeddings = document_embeddings[top_indices]

In [108]:
# Encode the BioASQ query using MedCPT Query Encoder

inputs = query_tokenizer(
    query,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = query_model(**inputs)

query_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

print(query_embedding.shape)

(1, 768)


In [109]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_embedding,
    bm25_embeddings
)[0]

print(similarities.shape)

(50,)


In [110]:
import numpy as np

reranked_indices = np.argsort(similarities)[::-1]

print(reranked_indices[:10])

[ 1  0 21 10  2 33 25 17 20  5]


In [111]:
print("=" * 80)
print("Hybrid Retrieval Results (BM25 → MedCPT)")
print("=" * 80)

for rank, idx in enumerate(reranked_indices[:10], start=1):

    doc = bm25_docs[idx]

    print(f"\nRank {rank}")
    print(f"MedCPT Similarity : {similarities[idx]:.4f}")
    print(f"PMID              : {doc['PMID']}")
    print(f"Title             : {doc['title']}")

Hybrid Retrieval Results (BM25 → MedCPT)

Rank 1
MedCPT Similarity : 0.6559
PMID              : 112034
Title             : Another family with purine neucleoside phosphorylase deficiency.

Rank 2
MedCPT Similarity : 0.6138
PMID              : 202219
Title             : Dietary and serum lipids in the multifactorial etiology of atherosclerosis.

Rank 3
MedCPT Similarity : 0.5993
PMID              : 125411
Title             : [Mannonidosis. Apropos of 5 cases].

Rank 4
MedCPT Similarity : 0.5981
PMID              : 198168
Title             : Studies on the metabolic defect in Broad-beta disease (hyperlipoproteinaemia type III).

Rank 5
MedCPT Similarity : 0.5980
PMID              : 103250
Title             : The Montgomery lecture, 1977. Curious colobomata.

Rank 6
MedCPT Similarity : 0.5957
PMID              : 100662
Title             : Intramural diverticulosis of the esophagus.

Rank 7
MedCPT Similarity : 0.5950
PMID              : 190952
Title             : Multicentric reticulohisti

In [112]:
hybrid_pmids = [bm25_docs[idx]["PMID"] for idx in reranked_indices[:10]]

hits = set(hybrid_pmids).intersection(set(ground_truth_pmids))

print("Retrieved PMIDs:", hybrid_pmids)
print("Ground Truth PMIDs:", ground_truth_pmids)
print("Correctly Retrieved:", len(hits))
print("Matching PMIDs:", hits)

Retrieved PMIDs: [112034, 202219, 125411, 198168, 103250, 100662, 190952, 106817, 155397, 114742]
Ground Truth PMIDs: [15858239, 20598273, 6650562, 12239580, 21995290, 23001136, 15617541, 8896569, 15829955]
Correctly Retrieved: 0
Matching PMIDs: set()


In [113]:
all_pmids = set(doc["PMID"] for doc in documents)

present = []
missing = []

for pmid in ground_truth_pmids:
    if pmid in all_pmids:
        present.append(pmid)
    else:
        missing.append(pmid)

print("Present:", present)
print("Missing:", missing)

Present: []
Missing: [15858239, 20598273, 6650562, 12239580, 21995290, 23001136, 15617541, 8896569, 15829955]


We'll recreate filter_data.ipynb.

In [114]:
# All PMIDs available in our 10k PubMed subset

available_pmids = set(doc["PMID"] for doc in documents)

print("Number of papers in our corpus:", len(available_pmids))

Number of papers in our corpus: 10000


In [115]:
answerable_questions = []
unanswerable_questions = []

for q in bioasq["questions"]:

    gt_pmids = [
        int(part)
        for url in q["documents"]
        for part in [url.split("/")[-1]]
        if part.isdigit()
    ]

    # Does at least ONE answer paper exist?
    if any(pmid in available_pmids for pmid in gt_pmids):
        answerable_questions.append(q)
    else:
        unanswerable_questions.append(q)

print("Total Questions       :", len(bioasq["questions"]))
print("Answerable Questions  :", len(answerable_questions))
print("Unanswerable Questions:", len(unanswerable_questions))

Total Questions       : 5389
Answerable Questions  : 1
Unanswerable Questions: 5388


In [116]:
import json

with open("filtered_bioasq.json", "w") as f:
    json.dump(
        {"questions": answerable_questions},
        f,
        indent=2
    )

print("Saved filtered dataset!")

Saved filtered dataset!


downloading and use precomputed medcpt embeddings dataset

In [117]:
!pip install --upgrade -q huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.26.0 which is incompatible.
transformers 4.40.0 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.26.0 which is incompatible.


In [118]:
!pip uninstall -y huggingface_hub
!pip install -q huggingface_hub==0.22.2

from huggingface_hub import login

login()

Found existing installation: huggingface_hub 1.26.0
Uninstalling huggingface_hub-1.26.0:
  Successfully uninstalled huggingface_hub-1.26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.22.2 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.22.2 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.22.2 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.22.2 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'get_token_permission' (from 'huggingface_hub.hf_api') is deprecated and will be removed from version '1.0'. Permissions are more complex than when `get_token_permission` was first introduced. OAuth and fine-grain tokens allows for more detailed permissions. If you need to know the permissions associated with a token, please use `whoami` and check the `'auth'` key.
  warnings.warn(warning_message, FutureWarning)


In [119]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="slinusc/PubMedAbstractsSubsetEmbedded",
    repo_type="dataset",
    filename="pubmed_chunk_0.jsonl"
)

print(file_path)

/root/.cache/huggingface/hub/datasets--slinusc--PubMedAbstractsSubsetEmbedded/snapshots/f24175696838635246ed7012181f923acd95a83b/pubmed_chunk_0.jsonl


In [120]:
import json

docs = []

with open(file_path, "r") as f:
    for i, line in enumerate(f):
        docs.append(json.loads(line))

        if i == 4:      # load first 5 documents only
            break

print("Loaded", len(docs), "documents")

Loaded 5 documents


In [121]:
docs[0]

{'title': '[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].',
 'abstract': 'A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without hyperglycemia in its most efficient dosage of 47 percent LD50 (150 

In [122]:
print(len(docs[0]["embedding"]))

768


In [123]:
print(docs[0].keys())

dict_keys(['title', 'abstract', 'PMID', 'embedding'])


In [124]:
import json
from tqdm import tqdm

documents = []

with open(file_path, "r") as f:
    for line in tqdm(f):
        documents.append(json.loads(line))

print("Documents:", len(documents))

100000it [00:19, 5226.62it/s]

Documents: 100000


In [125]:
import numpy as np

embeddings = np.array(
    [doc["embedding"] for doc in documents],
    dtype=np.float32
)

print(embeddings.shape)

(100000, 768)


In [126]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

In [128]:
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(768)

index.add(embeddings)

In [129]:
type(documents)
print(len(documents))
print(documents[0].keys())

100000
dict_keys(['title', 'abstract', 'PMID', 'embedding'])


In [130]:
print(type(embeddings))
print(isinstance(embeddings, np.ndarray))

print(embeddings.dtype if hasattr(embeddings, "dtype") else "No dtype")
print(embeddings.shape if hasattr(embeddings, "shape") else "No shape")

print(type(embeddings[0]))

<class 'numpy.ndarray'>
True
float32
(100000, 768)
<class 'numpy.ndarray'>


In [131]:
import numpy as np

embeddings = np.asarray(embeddings, dtype=np.float32)

print(type(embeddings))
print(embeddings.dtype)
print(embeddings.shape)

<class 'numpy.ndarray'>
float32
(100000, 768)


In [132]:
import faiss
print(faiss.__file__)
print(faiss.__version__ if hasattr(faiss, "__version__") else "No version")

/usr/local/lib/python3.12/dist-packages/faiss/__init__.py
1.8.0


In [133]:
import numpy as np
import faiss

# Make sure it's a contiguous float32 array
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)

print(embeddings.flags)

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [134]:
print(embeddings.flags)
print(embeddings.strides)
print(embeddings[:2])

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

(3072, 4)
[[-0.02109968  0.00287746 -0.03120272 ...  0.01515973  0.00824596
   0.01145889]
 [-0.01322427 -0.00301176 -0.00988564 ...  0.00221608  0.03053099
  -0.00114368]]


In [135]:
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(768)
index.add(embeddings)

print(index.ntotal)

100000


In [136]:
import numpy as np

faiss_embeddings = np.ascontiguousarray(embeddings, dtype='float32')
faiss.normalize_L2(faiss_embeddings)
index.add(faiss_embeddings)

query_vector = faiss_embeddings[0].reshape(1, -1)

scores, indices = index.search(query_vector, 5)

print(indices)
print(scores)

[[100000      0 119162  19162 114669]]
[[1.         1.         0.840364   0.840364   0.83226913]]


In [137]:
faiss.normalize_L2(embeddings)

In [138]:
index.add(embeddings)

In [139]:
print(index.ntotal)

300000


In [140]:
query_vector = embeddings[0].reshape(1,-1)

scores, indices = index.search(query_vector,5)

print(indices)
print(scores)

[[200000 100000      0 119162  19162]]
[[1.       1.       1.       0.840364 0.840364]]


In [141]:
question = bioasq["questions"][0]["body"]

print(question)

Is Hirschsprung disease a mendelian or a multifactorial disorder?


In [142]:
from transformers import AutoTokenizer, AutoModel

query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)

query_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
).to(device)

query_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [143]:
inputs = query_tokenizer(
    question,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    outputs = query_model(**inputs)

query_embedding = outputs.last_hidden_state[:,0,:]

query_embedding = query_embedding.cpu().numpy().astype(np.float32)

In [144]:
faiss.normalize_L2(query_embedding)

In [145]:
scores, indices = index.search(query_embedding,10)

In [149]:
for rank, idx in enumerate(indices[0], start=1):

    print("="*70)
    print("Rank:", rank)
    print("Similarity:", scores[0][rank-1])
    print("PMID:", documents[idx % len(documents)]["PMID"])
    print("Title:", documents[idx % len(documents)]["title"])

Rank: 1
Similarity: 0.7127273
PMID: 596109
Title: On the nature of syndrome delineation.
Rank: 2
Similarity: 0.7127273
PMID: 596109
Title: On the nature of syndrome delineation.
Rank: 3
Similarity: 0.71272725
PMID: 596109
Title: On the nature of syndrome delineation.
Rank: 4
Similarity: 0.6877356
PMID: 384265
Title: Clinical genetics and pediatric neoplasms: pathogenetic and etiologic perspectives.
Rank: 5
Similarity: 0.6877356
PMID: 384265
Title: Clinical genetics and pediatric neoplasms: pathogenetic and etiologic perspectives.
Rank: 6
Similarity: 0.6877356
PMID: 384265
Title: Clinical genetics and pediatric neoplasms: pathogenetic and etiologic perspectives.
Rank: 7
Similarity: 0.68061393
PMID: 381662
Title: Neural tube malformations: complex segregation analysis and calculation of recurrence risks.
Rank: 8
Similarity: 0.68061393
PMID: 381662
Title: Neural tube malformations: complex segregation analysis and calculation of recurrence risks.
Rank: 9
Similarity: 0.68061393
PMID: 38166

In [151]:
idx = indices[0][0]

print("PMID:", documents[idx % len(documents)]["PMID"])
print("Title:", documents[idx % len(documents)]["title"])
print()
print(documents[idx % len(documents)]["abstract"][:800])

PMID: 596109
Title: On the nature of syndrome delineation.

Perhaps half of all patients with multiple abnormalities have known, recognized syndromes. The other half represent unknown entities that need to be further delineated. The significance of syndrome delineation cannot be overestimated. As an unknown syndrome becomes delineated, its phenotypic spectrum, its natural history, and its risk of recurrence become known, allowing for better patient care and family counseling. The process of syndrome delineation is discussed in terms of unknown genesis syndromes of the previsionally-unique and recurrent-pattern types, and known-genesis syndromes of the pedigree, chromosomal, biochemical-defect, and environmentally-induced types. Several special syndrome categories are defined, including the chance syndrome, variant additive syndrome, association syn


# implementing the llm

In [152]:
!pip install -q google-generativeai

In [153]:
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [154]:
genai.configure(api_key="AQ.Ab8RN6LxIE-_gkHZlRHTS3W0diozOK6FKtdSusMzyFJRytDnuA")

In [162]:
model = genai.GenerativeModel("gemini-2.0-flash")

In [163]:
context = ""

# Construct retrieved_docs from the hybrid retrieval results
retrieved_docs = [bm25_docs[idx] for idx in reranked_indices]

for i, doc in enumerate(retrieved_docs, start=1):
    context += f"""
Document {i}

PMID: {doc['PMID']}

Title:
{doc['title']}

Abstract:
{doc['abstract']}

-----------------------------------
"""

In [164]:
prompt = f"""
You are an expert biomedical assistant.

Answer ONLY using the retrieved papers.

If the answer is not contained in the papers,
say:

"I could not find enough evidence."

Question:

{question}

Retrieved Papers:

{context}

Answer:
"""

In [160]:
import google.generativeai as genai

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [161]:
import google.generativeai as genai

print(genai.__version__)

0.8.6


In [167]:
import google.generativeai as genai

genai.configure(api_key="AQ.Ab8RN6LxIE-_gkHZlRHTS3W0diozOK6FKtdSusMzyFJRytDnuA")

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
response = model.generate_content(prompt)

print(response.text)